# Recursive Polynomial Basis — v2 (documented)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# ================= Recursive polynomial basis =================

class RecursivePolyBasis(nn.Module):
    """
    Learnable recursive polynomial basis for KAN edges:

        R_0(x) = 0
        R_1(x) = 1      (can be seen as a constant basis term)
        R_{n+1}(x) = (a x^2 + b x + c) R_n(x) + (d x + e) R_{n-1}(x)

    Edge output:
        y(x) = sum_{n=0}^K w_n R_n(x)

    a,b,c,d,e are shared across n and are learned from data.
    """
    def __init__(self, K, init_like_cheb=True, param_bound=3.0):
        super().__init__()
        self.K = K
        self.param_bound = param_bound

        # unconstrained parameters (squashed via tanh to keep them bounded)
        self.a_raw = nn.Parameter(torch.zeros(1))
        self.b_raw = nn.Parameter(torch.zeros(1))
        self.c_raw = nn.Parameter(torch.zeros(1))
        self.d_raw = nn.Parameter(torch.zeros(1))
        self.e_raw = nn.Parameter(torch.zeros(1))

        # weights for linear combination of basis functions
        self.w = nn.Parameter(torch.zeros(K + 1))

        if init_like_cheb:
            with torch.no_grad():
                # heuristics: make it vaguely look like Chebyshev: R_{n+1} ≈ 2x R_n - R_{n-1}
                self.a_raw.fill_(0.0)
                self.b_raw.fill_(0.5)   # tanh(0.5) ≈ 0.46 -> ~ 1.4 * x
                self.c_raw.fill_(0.0)
                self.d_raw.fill_(0.0)
                self.e_raw.fill_(-0.5)
                self.w.normal_(mean=0.0, std=0.01)

    def _squash_params(self):
        # map raw params to [-param_bound, param_bound]
        a = self.param_bound * torch.tanh(self.a_raw)
        b = self.param_bound * torch.tanh(self.b_raw)
        c = self.param_bound * torch.tanh(self.c_raw)
        d = self.param_bound * torch.tanh(self.d_raw)
        e = self.param_bound * torch.tanh(self.e_raw)
        return a, b, c, d, e

    def forward(self, x):
        """
        x: tensor of shape (batch,) or (batch,1)
        returns: (batch,)
        """
        if x.dim() > 1:
            x = x.squeeze(-1)

        # optional clipping for numerical stability
        x = x.clamp(-2.0, 2.0)

        a, b, c, d, e = self._squash_params()

        # initial polynomials
        R0 = torch.zeros_like(x)   # R_0(x)
        R1 = torch.ones_like(x)    # R_1(x)

        Rs = [R0, R1]

        # build R_2,...,R_K by recurrence
        for n in range(1, self.K):
            coef1 = a * x**2 + b * x + c
            coef2 = d * x + e
            R_next = coef1 * Rs[-1] + coef2 * Rs[-2]
            Rs.append(R_next)

        # shape: (batch, K+1)
        R_stack = torch.stack(Rs, dim=-1)

        # linear combination along basis index
        y = (R_stack * self.w).sum(dim=-1)
        return y

# ================= Simple 1D Poly-KAN model =================

class PolyKAN1D(nn.Module):
    """
    Minimal KAN-like 1D model: linear projection + recursive polynomial basis.
    """
    def __init__(self, K):
        super().__init__()
        self.lin = nn.Linear(1, 1)
        self.basis = RecursivePolyBasis(K=K, init_like_cheb=True)

    def forward(self, x):
        # x: (batch,1)
        h = self.lin(x)           # (batch,1)
        h = h.squeeze(-1)         # (batch,)
        y = self.basis(h)         # (batch,)
        return y.unsqueeze(-1)    # (batch,1)

# ================= Target function and data =================

def target_function(x):
    # x: (batch,1)
    return x**2 + torch.sin(x)

def make_dataset(n_samples, device):
    # sample uniformly on [-2, 2]
    x = 4.0 * torch.rand(n_samples, 1, device=device) - 2.0
    y = target_function(x)
    return x, y

# ================= Training and evaluation =================

def train_and_eval():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # hyperparameters
    K = 8
    n_samples = 2000
    n_epochs = 2000
    batch_size = 256
    lr = 1e-3

    # data
    x_all, y_all = make_dataset(n_samples, device)

    model = PolyKAN1D(K=K).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    for epoch in range(1, n_epochs + 1):
        perm = torch.randperm(n_samples, device=device)
        x_all = x_all[perm]
        y_all = y_all[perm]

        for i in range(0, n_samples, batch_size):
            x_batch = x_all[i:i+batch_size]
            y_batch = y_all[i:i+batch_size]

            pred = model(x_batch)
            loss = loss_fn(pred, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if epoch % 200 == 0:
            print(f"Epoch {epoch}/{n_epochs}, loss = {loss.item():.6f}")

    # print learned parameters
    basis = model.basis
    a, b, c, d, e = basis._squash_params()
    print("\nLearned recurrence parameters (after tanh squash):")
    print(f"a = {a.item():.4f}, b = {b.item():.4f}, c = {c.item():.4f}, d = {d.item():.4f}, e = {e.item():.4f}")
    print("w =", basis.w.data.cpu().numpy())

    # evaluate on a dense grid for plotting
    x_test = torch.linspace(-2, 2, 400, device=device).unsqueeze(-1)
    with torch.no_grad():
        y_pred = model(x_test)
    y_true = target_function(x_test)

    # to numpy
    x_np = x_test.cpu().numpy().squeeze()
    y_pred_np = y_pred.cpu().numpy().squeeze()
    y_true_np = y_true.cpu().numpy().squeeze()

    # plot
    plt.figure(figsize=(6, 4))
    plt.plot(x_np, y_true_np, label='x^2 + sin(x)', color='black')
    plt.plot(x_np, y_pred_np, label='RecursivePoly-KAN approx', color='red', linestyle='--')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    train_and_eval()
